In [1]:
import pandas as pd
import numpy as np


In [8]:
df =  pd.read_parquet("../data/raw/yellow/yellow_tripdata_2025-01.parquet")

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [15]:
print (f"Pick up date range: {df['tpep_pickup_datetime'].min()} - {df['tpep_pickup_datetime'].max()} ({df['tpep_pickup_datetime'].max() - df['tpep_pickup_datetime'].min()})")
print (f"Drop off date range: {df['tpep_dropoff_datetime'].min()} - {df['tpep_dropoff_datetime'].max()} ({df['tpep_dropoff_datetime'].max() - df['tpep_dropoff_datetime'].min()})")

Pick up date range: 2024-12-31 20:47:55 - 2025-02-01 00:00:44 (31 days 03:12:49)
Drop off date range: 2024-12-18 07:52:40 - 2025-02-01 23:44:11 (45 days 15:51:31)


In [18]:
df.isna().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64

In [55]:
#Drop of before pickup 
df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]['tpep_dropoff_datetime'].count()

np.int64(124)

In [56]:
#Zero Duration
df[df['tpep_dropoff_datetime'] == df['tpep_pickup_datetime']]['tpep_dropoff_datetime'].count()

np.int64(1927)

In [57]:
#Impossible trip
df[df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']]['tpep_dropoff_datetime'].count()

np.int64(2051)

In [60]:
#Trip distance range 
print(f"{df['trip_distance'].max()} - {df['trip_distance'].min()}")

276423.57 - 0.0


In [82]:
#zero distance trip
df[df['trip_distance'] == 0]['trip_distance'].count()

np.int64(90893)

In [61]:
#Fare amount range
print(f"{df['fare_amount'].max()} - {df['fare_amount'].min()}")

863372.12 - -900.0


In [62]:
#total amount range
print(f"{df['total_amount'].max()} - {df['total_amount'].min()}")

863380.37 - -901.0


In [64]:
#Passenger range
print(f"{df['passenger_count'].max()} - {df['passenger_count'].min()}")

9.0 - 0.0


In [68]:
#Passenger count = 0 trips
df[df['passenger_count'] == 0]['passenger_count'].count()

np.int64(24656)

In [69]:
df['PULocationID'].isna().sum()

np.int64(0)

In [70]:
df['DOLocationID'].isna().sum()

np.int64(0)

In [4]:
df.duplicated().sum()

np.int64(0)

In [9]:
#Flagging suspisous data
df['flag_extreme_dist'] = df['trip_distance'] >= 50

duration_hours =( df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() /  3600
speed_mph = df['trip_distance'] / duration_hours.replace(0, np.nan)
df["flag_impossible_speed"] = ((speed_mph > 80) | (speed_mph < 0) | speed_mph.isna())

print(df['flag_extreme_dist'].sum())
print(df['flag_impossible_speed'].sum())

534
3033


In [10]:
clean_mask = ((df['tpep_dropoff_datetime'] >= df['tpep_pickup_datetime'])
    &
    (df['trip_distance'] >= 0)
    &
    (df['fare_amount'] >= 0)
    &
    (df['total_amount'] >= 0)
    &
    (~df['PULocationID'].isna()))

clean_yellow = df[clean_mask].copy()
rejected_yellow = df[~clean_mask].copy()



In [11]:
summary = {
    'yellow_clean':len(clean_yellow),
    'yellow_rejected':len(rejected_yellow),
    'total': len(clean_yellow) + len(rejected_yellow),
    'raw len': len(df)
}
summary


{'yellow_clean': 3330663,
 'yellow_rejected': 144563,
 'total': 3475226,
 'raw len': 3475226}

In [12]:
clean_yellow = clean_yellow.drop(columns = [
        'store_and_fwd_flag',
        'VendorID',
        'mta_tax',
        'improvement_surcharge',
    ])

In [13]:
#Derived columns


clean_yellow['pickup_date'] = clean_yellow['tpep_pickup_datetime'].dt.date
clean_yellow['pickup_hour'] = clean_yellow['tpep_pickup_datetime'].dt.hour
clean_yellow['pickup_dayofw'] = clean_yellow['tpep_pickup_datetime'].dt.day_of_week
clean_yellow['pickup_month'] = clean_yellow['tpep_pickup_datetime'].dt.to_period('M')

duration = ( clean_yellow['tpep_dropoff_datetime'] - clean_yellow['tpep_pickup_datetime']).dt.total_seconds()
clean_yellow['duration_mins'] = duration / 60
clean_yellow['speed_mph']= clean_yellow['trip_distance'] / duration / 3600

clean_yellow['tip_rate'] = clean_yellow['tip_amount'] / clean_yellow['fare_amount'].replace(0, np.nan)
clean_yellow['fare_per_mile'] = clean_yellow['fare_amount'] / clean_yellow['trip_distance']
clean_yellow['rev_per_min'] = clean_yellow['total_amount'] / clean_yellow['duration_mins']


clean_yellow.head()


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,flag_impossible_speed,pickup_date,pickup_hour,pickup_dayofw,pickup_month,duration_mins,speed_mph,tip_rate,fare_per_mile,rev_per_min
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,229,237,1,10.0,3.5,...,False,2025-01-01,0,2,2025-01,8.350000,8.871147e-07,0.300000,6.250000,2.155689
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,236,237,1,5.1,3.5,...,False,2025-01-01,0,2,2025-01,2.550000,9.077705e-07,0.396078,10.200000,4.752941
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,141,141,1,5.1,3.5,...,False,2025-01-01,0,2,2025-01,1.950000,1.424501e-06,0.392157,8.500000,6.205128
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,244,244,2,7.2,1.0,...,False,2025-01-01,0,2,2025-01,5.566667,4.324684e-07,0.000000,13.846154,1.742515
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,244,116,2,5.8,1.0,...,False,2025-01-01,0,2,2025-01,3.533333,8.647799e-07,0.000000,8.787879,2.349057


In [6]:
clean_yellow.to_parquet("../data/processed/clean_yellow_2025-01.parquet")

In [14]:
#Export data
clean_yellow.to_parquet("../data/processed/clean_yellow_2025-01.parquet")
rejected_yellow.to_parquet("../data/processed/rejected_yellow_2025-01.parquet")

In [15]:
#Start create marts
#Use this to load clean data

df = pd.read_parquet("../data/processed/clean_yellow_2025-01.parquet")

df.head()


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,flag_impossible_speed,pickup_date,pickup_hour,pickup_dayofw,pickup_month,duration_mins,speed_mph,tip_rate,fare_per_mile,rev_per_min
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,229,237,1,10.0,3.5,...,False,2025-01-01,0,2,2025-01,8.350000,8.871147e-07,0.300000,6.250000,2.155689
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,236,237,1,5.1,3.5,...,False,2025-01-01,0,2,2025-01,2.550000,9.077705e-07,0.396078,10.200000,4.752941
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,141,141,1,5.1,3.5,...,False,2025-01-01,0,2,2025-01,1.950000,1.424501e-06,0.392157,8.500000,6.205128
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,244,244,2,7.2,1.0,...,False,2025-01-01,0,2,2025-01,5.566667,4.324684e-07,0.000000,13.846154,1.742515
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,244,116,2,5.8,1.0,...,False,2025-01-01,0,2,2025-01,3.533333,8.647799e-07,0.000000,8.787879,2.349057


In [18]:
#Hourly mart demand
hourly_mart = (
    df
    .groupby("pickup_hour")
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values("pickup_hour", ascending = True)
)

hourly_mart['avg_tip_rate'] = hourly_mart['avg_tip_rate'] * 100


In [20]:
#Daily mart demand
daily_mart = (
    df
    .groupby("pickup_dayofw")
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values("pickup_dayofw", ascending = True)
)
daily_mart['avg_tip_rate'] = daily_mart['avg_tip_rate'] * 100

daily_mart

,total_passenger,trip_count,avg_trip_distance,revenue,avg_tip_rate,avg_duration
pickup_dayofw,,,,,,
0,387884.0,350271,4.190123,10739182.86,21.222447,14.930152
1,465709.0,437218,5.213843,11669784.76,20.860431,15.167001
2,604213.0,550321,5.309459,14728913.95,20.321588,15.193285
3,642112.0,590000,5.971884,16086115.25,19.997138,15.783780
4,633778.0,562235,5.678457,15208344.45,19.458600,15.646378
5,556724.0,456636,4.780722,11545270.80,19.902870,14.317191
6,436821.0,383982,6.184170,10355598.40,18.437940,13.718678


In [22]:
#Pick up zone demand
pickup_demand = (
    df.groupby('PULocationID')
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values('trip_count', ascending = False)
)
pickup_demand['avg_tip_rate'] = pickup_demand['avg_tip_rate'] * 100



In [24]:
#Drop off zone demand
dropoff_common = (
    df.groupby('DOLocationID')
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values('trip_count', ascending = False)
)
dropoff_common['avg_tip_rate'] = dropoff_common['avg_tip_rate'] * 100

In [26]:
#Payment Tip behavior

tip_behavior = (
    df.groupby('payment_type')
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values('avg_tip_rate', ascending = False)
)
tip_behavior['avg_tip_rate'] = tip_behavior['avg_tip_rate'] * 100

In [28]:
#Trip distance distribution
bins = [0,1,3,7,15, np.inf]
labels = ["0-1mi",'1-3mi','3-5mi','7-15mi','15+mi']

df['distance_group'] = pd.cut(df['trip_distance'],bins = bins,labels = labels, right = False)

trip_distribution = (
    df.groupby('distance_group')
    .agg(
        total_passenger = ('passenger_count','sum'),
        trip_count = ('tpep_pickup_datetime','count'),
        avg_trip_distance =('trip_distance','mean'),
        revenue= ("total_amount","sum"),
        avg_tip_rate = ('tip_rate','mean'),
        avg_duration = ('duration_mins','mean')
    )
    .sort_values('trip_count', ascending = False)
)
trip_distribution['avg_tip_rate'] = trip_distribution['avg_tip_rate'] * 100


C:\Users\Loc Nguyen\AppData\Local\Temp\ipykernel_19772\3933700425.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('distance_group')


In [ ]:
#Abnormal mart
abnormal_mart =df[ df['flag_impossible_speed'] | df['flag_extreme_dist']].copy()

In [81]:
hourly_mart.to_parquet('../data/marts/mart_hourly_demand.parquet')
daily_mart.to_parquet('../data/marts/mart_daily_demand.parquet')
pickup_demand.to_parquet('../data/marts/mart_pickup_demand.parquet')
dropoff_common.to_parquet('../data/marts/mart_dropoff_common.parquet')
tip_behavior.to_parquet('../data/marts/mart_tip_behavior.parquet')
trip_distribution.to_parquet('../data/marts/mart_trip_distribution.parquet')
abnormal_mart .to_parquet('../data/marts/mart_abnormals.parquet')
